In [ ]:
!pip install -q ultralytics==8.4.66 onnx onnxruntime onnxslim pyyaml

In [ ]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")

print("Kaggle 输入目录：")
for path in sorted(input_root.rglob("*")):
    if path.is_file():
        print(path)

In [ ]:
from pathlib import Path
import shutil

input_root = Path("/kaggle/input")
target_root = Path("/kaggle/working/dataset")

# 自动寻找真正包含 images、labels、data.yaml 的目录
candidates = []

for yaml_path in input_root.rglob("data.yaml"):
    root = yaml_path.parent

    if (
        (root / "images" / "train").exists()
        and (root / "images" / "val").exists()
        and (root / "images" / "test").exists()
        and (root / "labels" / "train").exists()
        and (root / "labels" / "val").exists()
        and (root / "labels" / "test").exists()
    ):
        candidates.append(root)

print("找到的数据集：")
for path in candidates:
    print(path)

if len(candidates) == 0:
    raise FileNotFoundError("没有找到完整的 YOLO 数据集目录")

if len(candidates) > 1:
    raise RuntimeError(f"找到多个数据集目录：{candidates}")

source_root = candidates[0]

# 清理 working 中之前可能残留的数据
if target_root.exists():
    shutil.rmtree(target_root)

# input 为只读，复制到 working
shutil.copytree(source_root, target_root)

print("\n源目录：", source_root)
print("目标目录：", target_root)
print("data.yaml：", (target_root / "data.yaml").exists())
print("训练图片目录：", (target_root / "images" / "train").exists())
print("训练标签目录：", (target_root / "labels" / "train").exists())

In [ ]:
from pathlib import Path

dataset_root = Path("/kaggle/working/dataset")

data_yaml = """path: /kaggle/working/dataset

train: images/train
val: images/val
test: images/test

names:
  0: red_signal
  1: yellow_signal
  2: green_signal
  3: puddle
  4: closed_manhole
  5: ban
  6: slippery
  7: walk
"""

yaml_path = dataset_root / "data.yaml"
yaml_path.write_text(data_yaml, encoding="utf-8")

print(yaml_path.read_text(encoding="utf-8"))

In [ ]:
from pathlib import Path
from collections import Counter

dataset_root = Path("/kaggle/working/dataset")

image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

class_names = {
    0: "red_signal",
    1: "yellow_signal",
    2: "green_signal",
    3: "puddle",
    4: "closed_manhole",
    5: "ban",
    6: "slippery",
    7: "walk",
}

all_errors = []
split_statistics = {}

for split in ["train", "val", "test"]:
    image_dir = dataset_root / "images" / split
    label_dir = dataset_root / "labels" / split

    images = sorted([
        path
        for path in image_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() in image_extensions
    ])

    labels = sorted([
        path
        for path in label_dir.iterdir()
        if path.is_file()
        and path.suffix.lower() == ".txt"
        and path.name.lower() != "classes.txt"
    ])

    image_stems = {path.stem for path in images}
    label_stems = {path.stem for path in labels}

    missing_labels = image_stems - label_stems
    orphan_labels = label_stems - image_stems

    class_counter = Counter()
    empty_labels = 0

    for label_path in labels:
        text = label_path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip()

        if not text:
            empty_labels += 1
            continue

        for line_number, line in enumerate(
            text.splitlines(),
            start=1
        ):
            parts = line.split()

            if len(parts) != 5:
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"第 {line_number} 行不是 5 列"
                )
                continue

            try:
                class_id = int(parts[0])
                xc, yc, width, height = map(float, parts[1:])
            except ValueError:
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"第 {line_number} 行无法解析"
                )
                continue

            if class_id not in class_names:
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"非法 class_id={class_id}"
                )

            if not all(
                0.0 <= value <= 1.0
                for value in [xc, yc, width, height]
            ):
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"坐标不在 0~1"
                )

            if width <= 0 or height <= 0:
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"宽高小于等于 0"
                )

            x1 = xc - width / 2
            y1 = yc - height / 2
            x2 = xc + width / 2
            y2 = yc + height / 2

            if (
                x1 < -1e-6
                or y1 < -1e-6
                or x2 > 1 + 1e-6
                or y2 > 1 + 1e-6
            ):
                all_errors.append(
                    f"{split}/{label_path.name} "
                    f"检测框越界"
                )

            class_counter[class_id] += 1

    if missing_labels:
        all_errors.append(
            f"{split} 有 {len(missing_labels)} 张图片缺少标签"
        )

    if orphan_labels:
        all_errors.append(
            f"{split} 有 {len(orphan_labels)} 个标签缺少图片"
        )

    split_statistics[split] = {
        "images": len(images),
        "labels": len(labels),
        "empty_labels": empty_labels,
        "class_counter": class_counter,
    }

for split, stats in split_statistics.items():
    print(f"\n========== {split} ==========")
    print("图片数量：", stats["images"])
    print("标签数量：", stats["labels"])
    print("空标签数量：", stats["empty_labels"])

    print("类别实例数：")
    for class_id, class_name in class_names.items():
        print(
            class_id,
            class_name,
            stats["class_counter"][class_id]
        )

print("\n========== 检查结果 ==========")

if all_errors:
    print("发现问题数量：", len(all_errors))
    for error in all_errors[:100]:
        print(error)

    raise RuntimeError("数据集存在问题，请停止训练。")
else:
    print("未发现格式问题，可以训练。")

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

dataset_root = Path("/kaggle/working/dataset")

class_names = {
    0: "red_signal",
    1: "yellow_signal",
    2: "green_signal",
    3: "puddle",
    4: "closed_manhole",
    5: "ban",
    6: "slippery",
    7: "walk",
}

image_dir = dataset_root / "images" / "train"
label_dir = dataset_root / "labels" / "train"

image_paths = [
    path
    for path in image_dir.iterdir()
    if path.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp"
    }
]

random.seed(42)
sample_paths = random.sample(
    image_paths,
    min(12, len(image_paths))
)

plt.figure(figsize=(16, 12))

for index, image_path in enumerate(sample_paths, start=1):
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    label_path = label_dir / f"{image_path.stem}.txt"

    if label_path.exists():
        lines = label_path.read_text(
            encoding="utf-8",
            errors="ignore"
        ).strip().splitlines()

        for line in lines:
            parts = line.split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])
            xc, yc, box_w, box_h = map(float, parts[1:])

            x1 = int((xc - box_w / 2) * width)
            y1 = int((yc - box_h / 2) * height)
            x2 = int((xc + box_w / 2) * width)
            y2 = int((yc + box_h / 2) * height)

            cv2.rectangle(
                image,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            cv2.putText(
                image,
                class_names.get(class_id, str(class_id)),
                (max(0, x1), max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 0, 0),
                2
            )

    plt.subplot(3, 4, index)
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

train_results = model.train(
    data="/kaggle/working/dataset/data.yaml",

    epochs=120,
    imgsz=640,
    batch=16,

    device=0,
    workers=2,

    patience=30,
    cos_lr=True,
    close_mosaic=10,

    amp=True,
    seed=42,
    deterministic=True,

    project="/kaggle/working/runs",
    name="smart_cane_yolov8n_v1",

    save=True,
    plots=True
)

In [ ]:
from pathlib import Path

run_dir = Path(
    "/kaggle/working/runs/smart_cane_yolov8n_v1"
)

best_path = run_dir / "weights" / "best.pt"
last_path = run_dir / "weights" / "last.pt"

print("best.pt exists:", best_path.exists())
print("best.pt:", best_path)

print("last.pt exists:", last_path.exists())
print("last.pt:", last_path)

print("\n训练目录文件：")

for path in sorted(run_dir.rglob("*")):
    if path.is_file():
        print(path)

In [ ]:
from ultralytics import YOLO

best_path = (
    "/kaggle/working/runs/"
    "smart_cane_yolov8n_v1/weights/best.pt"
)

best = YOLO(best_path)

val_metrics = best.val(
    data="/kaggle/working/dataset/data.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True
)

In [ ]:
test_metrics = best.val(
    data="/kaggle/working/dataset/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    plots=True,

    project="/kaggle/working/evaluation",
    name="smart_cane_test_v1"
)

In [ ]:
prediction_results = best.predict(
    source="/kaggle/working/dataset/images/test",

    imgsz=640,
    conf=0.25,
    iou=0.7,

    device=0,

    save=True,
    save_txt=False,

    project="/kaggle/working/predictions",
    name="smart_cane_test_predictions"
)

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

prediction_dir = Path(
    "/kaggle/working/predictions/"
    "smart_cane_test_predictions"
)

prediction_images = sorted([
    path
    for path in prediction_dir.iterdir()
    if path.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png"
    }
])

print("预测图片数量：", len(prediction_images))

for image_path in prediction_images[:20]:
    print(image_path.name)
    display(Image.open(image_path))

In [ ]:
from pathlib import Path
import shutil

best_source = Path(
    "/kaggle/working/runs/"
    "smart_cane_yolov8n_v1/weights/best.pt"
)

best_output = Path(
    "/kaggle/working/smart_cane_yolov8n_v1_best.pt"
)

shutil.copy2(best_source, best_output)

print("已复制：", best_output)
print("文件大小：", best_output.stat().st_size / 1024**2, "MB")

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

best = YOLO(
    "/kaggle/working/"
    "smart_cane_yolov8n_v1_best.pt"
)

export_result = best.export(
    format="onnx",
    imgsz=640,
    opset=11,
    simplify=True,
    dynamic=False,
    nms=False
)

print("导出结果：", export_result)

onnx_source = Path(export_result)

onnx_output = Path(
    "/kaggle/working/"
    "smart_cane_yolov8n_v1.onnx"
)

if onnx_source.resolve() != onnx_output.resolve():
    shutil.copy2(onnx_source, onnx_output)

print("最终 ONNX：", onnx_output)

In [ ]:
from pathlib import Path
import shutil

package_dir = Path(
    "/kaggle/working/"
    "smart_cane_yolov8n_v1_artifacts"
)

if package_dir.exists():
    shutil.rmtree(package_dir)

package_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    Path(
        "/kaggle/working/"
        "smart_cane_yolov8n_v1_best.pt"
    ),
    Path(
        "/kaggle/working/"
        "smart_cane_yolov8n_v1.onnx"
    ),
    Path(
        "/kaggle/working/runs/"
        "smart_cane_yolov8n_v1/results.png"
    ),
    Path(
        "/kaggle/working/runs/"
        "smart_cane_yolov8n_v1/results.csv"
    ),
    Path(
        "/kaggle/working/runs/"
        "smart_cane_yolov8n_v1/"
        "confusion_matrix.png"
    ),
    Path(
        "/kaggle/working/runs/"
        "smart_cane_yolov8n_v1/"
        "confusion_matrix_normalized.png"
    ),
    Path(
        "/kaggle/working/dataset/data.yaml"
    ),
]

for source in files_to_copy:
    if source.exists():
        shutil.copy2(source, package_dir / source.name)
        print("已复制：", source.name)
    else:
        print("未找到：", source)

zip_path = shutil.make_archive(
    "/kaggle/working/"
    "smart_cane_yolov8n_v1_artifacts",
    "zip",
    package_dir
)

print("结果压缩包：", zip_path)